# Train Terms Mismatch EDA

This notebook is a focused diagnostic task for the current `Train/train_terms.tsv` mismatch.

Goals:
- materialize the full pair-level mismatch inventory
- enrich mismatches with recreated-source Swiss-Prot evidence metadata
- write targeted raw Swiss-Prot text snippets for manual inspection
- identify the dominant root-cause categories before changing the main train pipeline


## Setup
This notebook is intentionally notebook-local and diagnostic-specific. It should not mutate the main pipeline logic.


In [ ]:
from __future__ import annotations

from collections import Counter
from datetime import date
from io import StringIO
from pathlib import Path
from pprint import pprint
import gzip
import sys

import pandas as pd
from Bio import SwissProt

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
from cafa import ProjectConfig
from cafa.io import read_train_taxonomy_rows, read_train_term_rows
from cafa.ontology import canonicalize_go_id, read_go_obo
from cafa.sources import download_source, extract_tar_gz_member, resolve_uniprot_sprot_snapshot
from cafa.validation import filter_reference_train_taxonomy_rows, filter_reference_train_term_rows

## Configure the same run scope as the main pipeline
Keep this config aligned with the last `main.ipynb` run whose recreated artifacts are being diagnosed.


In [ ]:
config = ProjectConfig(
    go_release="2025-06-01",
    train_uniprot_release="2025_03",
    submission_deadline=date(2026, 1, 27),
    evaluation_time=date(2026, 3, 17),
    train_taxon_ids=(
        9606, 10090, 3702, 559292, 10116, 284812, 83333, 7227,
        6239, 83332, 7955, 44689, 39947, 9913, 9031, 8355, 237561,
    ),
    test_taxon_ids=(),
    subontologies=("MF", "BP", "CC"),
    evidence_codes=(
        "EXP", "IDA", "IPI", "IMP", "IGI", "IEP", "HTP",
        "HDA", "HMP", "HGI", "HEP", "TAS", "IC",
    ),
    similarity_backend="diamond",
    validation_mode="canonical",
    project_root=PROJECT_ROOT,
    cache_dir=Path(".cache/cafa"),
    recreated_data_dir=Path("recreated_comp_data"),
    artifacts_dir=Path("artifacts"),
    results_dir=Path("results"),
)

output_dir = PROJECT_ROOT / "results" / "train_terms_diagnostics"
output_dir.mkdir(parents=True, exist_ok=True)

pprint(
    {
        "train_taxon_ids": config.train_taxon_ids,
        "subontologies": config.subontologies,
        "evidence_codes": config.evidence_codes,
        "output_dir": str(output_dir),
    }
)


## Resolve required artifacts
This notebook reads already-created recreated artifacts and uses the pinned Swiss-Prot flatfile only for diagnostics.


In [ ]:
ontology_path = PROJECT_ROOT / "recreated_comp_data" / "Train" / "go-basic.obo"
recreated_taxonomy_path = PROJECT_ROOT / "recreated_comp_data" / "Train" / "train_taxonomy.tsv"
recreated_terms_path = PROJECT_ROOT / "recreated_comp_data" / "Train" / "train_terms.tsv"
reference_terms_path = PROJECT_ROOT / "comp_data" / "Train" / "train_terms.tsv"

ontology = read_go_obo(ontology_path)
allowed_taxon_ids = set(config.train_taxon_ids)
allowed_subontologies = set(config.subontologies)
allowed_evidence_codes = set(config.evidence_codes)

uniprot_snapshot = resolve_uniprot_sprot_snapshot(config)
flatfile_gz_path = extract_tar_gz_member(
    download_source(uniprot_snapshot),
    "uniprot_sprot.dat.gz",
)

required_paths = (
    ontology_path,
    recreated_taxonomy_path,
    recreated_terms_path,
    reference_terms_path,
    flatfile_gz_path,
)
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

pprint(
    {
        "ontology_path": str(ontology_path),
        "recreated_taxonomy_path": str(recreated_taxonomy_path),
        "recreated_terms_path": str(recreated_terms_path),
        "reference_terms_path": str(reference_terms_path),
        "flatfile_gz_path": str(flatfile_gz_path),
    }
)

## Load filtered recreated and reference rows
The filtering scope matches the main validator: selected train taxa and selected subontologies.


In [ ]:
taxonomy_rows = filter_reference_train_taxonomy_rows(
    read_train_taxonomy_rows(recreated_taxonomy_path),
    allowed_taxon_ids=allowed_taxon_ids,
)
allowed_protein_ids = {row.protein_id for row in taxonomy_rows}
taxon_by_protein = {row.protein_id: row.taxon_id for row in taxonomy_rows}

recreated_rows = filter_reference_train_term_rows(
    read_train_term_rows(recreated_terms_path, ontology=ontology),
    allowed_protein_ids=allowed_protein_ids,
    allowed_subontologies=allowed_subontologies,
)
reference_rows = filter_reference_train_term_rows(
    read_train_term_rows(reference_terms_path, ontology=ontology),
    allowed_protein_ids=allowed_protein_ids,
    allowed_subontologies=allowed_subontologies,
)

print(
    {
        "taxonomy_rows": len(taxonomy_rows),
        "allowed_proteins": len(allowed_protein_ids),
        "recreated_term_rows": len(recreated_rows),
        "reference_term_rows": len(reference_rows),
    }
)

In [ ]:
# TODO: we can use _map_term_pairs_to_aspect from cafa.validation. this is the same exact logic
def map_term_pairs_to_aspect(rows):
    pair_to_aspect = {}
    for row in rows:
        key = (row.protein_id, row.term_id)
        existing_aspect = pair_to_aspect.get(key)
        if existing_aspect is not None and existing_aspect != row.aspect:
            raise ValueError(
                f"Conflicting aspects for {row.protein_id} / {row.term_id}: {existing_aspect} vs {row.aspect}"
            )
        pair_to_aspect[key] = row.aspect
    return pair_to_aspect


recreated_pair_map = map_term_pairs_to_aspect(recreated_rows)
reference_pair_map = map_term_pairs_to_aspect(reference_rows)

left_only_keys = sorted(set(recreated_pair_map) - set(reference_pair_map))
right_only_keys = sorted(set(reference_pair_map) - set(recreated_pair_map))
shared_mismatch_keys = sorted(
    key
    for key in set(recreated_pair_map) & set(reference_pair_map)
    if recreated_pair_map[key] != reference_pair_map[key]
)

print(
    {
        "left_only_count": len(left_only_keys),
        "right_only_count": len(right_only_keys),
        "shared_mismatch_count": len(shared_mismatch_keys),
    }
)

## Materialize the full mismatch inventory
The comparison unit here is the exact `(protein_id, canonical_go_id)` pair.


In [ ]:
def mismatch_records(keys, category):
    records = []
    for protein_id, term_id in keys:
        records.append(
            {
                "category": category,
                "protein_id": protein_id,
                "term_id": term_id,
                "recreated_aspect": recreated_pair_map.get((protein_id, term_id)),
                "reference_aspect": reference_pair_map.get((protein_id, term_id)),
                "taxon_id": taxon_by_protein.get(protein_id),
            }
        )
    return records

left_only_df = pd.DataFrame(mismatch_records(left_only_keys, "left_only"))
right_only_df = pd.DataFrame(mismatch_records(right_only_keys, "right_only"))
shared_mismatch_df = pd.DataFrame(mismatch_records(shared_mismatch_keys, "shared_mismatch"))
all_mismatches_df = pd.concat(
    [left_only_df, right_only_df, shared_mismatch_df],
    ignore_index=True,
)

print(all_mismatches_df.shape)
all_mismatches_df.head()

In [ ]:
all_mismatches_df['taxon_id'].value_counts()

## Scan Swiss-Prot only for mismatch proteins
This pass enriches mismatches with recreated-source evidence and captures raw record text for manual inspection.


In [ ]:
mismatch_protein_ids = set(all_mismatches_df["protein_id"])


def iter_swissprot_blocks(flatfile_path):
    buffer = []
    with gzip.open(flatfile_path, mode="rt", encoding="utf-8") as handle:
        for line in handle:
            buffer.append(line)
            if line.rstrip("\n") == "//":
                yield "".join(buffer)
                buffer = []
    if buffer:
        yield "".join(buffer)

In [ ]:
pair_source_metadata = {}
raw_text_by_protein = {}
scanned_mismatch_proteins = set()

for block in iter_swissprot_blocks(flatfile_gz_path):
    record = SwissProt.read(StringIO(block))
    if not record.accessions:
        continue
    protein_id = record.accessions[0]
    if protein_id not in mismatch_protein_ids:
        continue

    scanned_mismatch_proteins.add(protein_id)
    # TODO: what if there's already a block for a protein??
    raw_text_by_protein[protein_id] = block

    for reference in record.cross_references:
        if not reference or str(reference[0]).strip().upper() != "GO":
            continue
        if len(reference) < 4:
            continue

        raw_term_id = str(reference[1]).strip()
        canonical_term_id = canonicalize_go_id(ontology, raw_term_id)
        evidence_code = str(reference[3]).split(":", 1)[0].strip().upper()
        key = (protein_id, canonical_term_id)
        metadata = pair_source_metadata.setdefault(
            key,
            {
                "evidence_codes": set(),
                "raw_term_ids": set(),
                "has_allowed_evidence": False,
                "has_disallowed_evidence": False,
            },
        )
        metadata["raw_term_ids"].add(raw_term_id)
        if evidence_code:
            metadata["evidence_codes"].add(evidence_code)
            if evidence_code in allowed_evidence_codes:
                metadata["has_allowed_evidence"] = True
            else:
                metadata["has_disallowed_evidence"] = True

print(
    {
        "mismatch_proteins": len(mismatch_protein_ids),
        "mismatch_proteins_found_in_swissprot": len(scanned_mismatch_proteins),
        "pair_source_metadata_entries": len(pair_source_metadata),
    }
)

In [ ]:
def probable_reason(category, source_presence, has_allowed_evidence, has_disallowed_evidence):
    if category == "right_only":
        if not source_presence:
            return "absent_from_swissprot_go_refs"
        if has_allowed_evidence:
            return "present_with_allowed_evidence_but_missing_from_recreated"
        if has_disallowed_evidence:
            return "present_with_disallowed_evidence_only"
        return "present_without_parseable_evidence"

    if category == "left_only":
        if not source_presence:
            return "unexpected_recreated_pair_absent_from_swissprot_index"
            # TODO: this should not happen!! and what if it does?
        if has_allowed_evidence:
            return "recreated_only_pair_present_with_allowed_evidence"
        if has_disallowed_evidence:
            return "recreated_only_pair_present_with_disallowed_evidence"
        return "recreated_only_pair_present_without_parseable_evidence"

    if not source_presence:
        return "shared_pair_missing_from_swissprot_index"
    return "shared_pair_mapping_value_mismatch"


def enrich_mismatch_frame(frame):
    if frame.empty:
        return frame.assign(
            source_presence=pd.Series(dtype="object"),
            source_evidence_codes=pd.Series(dtype="object"),
            source_raw_term_ids=pd.Series(dtype="object"),
            source_has_allowed_evidence=pd.Series(dtype="bool"),
            source_has_disallowed_evidence=pd.Series(dtype="bool"),
            probable_reason=pd.Series(dtype="object"),
        )

    enriched_rows = []
    for row in frame.to_dict(orient="records"):
        key = (row["protein_id"], row["term_id"])
        metadata = pair_source_metadata.get(key)
        source_presence = metadata is not None
        evidence_codes = tuple(sorted(metadata["evidence_codes"])) if metadata else ()
        raw_term_ids = tuple(sorted(metadata["raw_term_ids"])) if metadata else ()
        has_allowed_evidence = bool(metadata and metadata["has_allowed_evidence"])
        has_disallowed_evidence = bool(metadata and metadata["has_disallowed_evidence"])
        row.update(
            {
                "source_presence": "present" if source_presence else "absent",
                "source_evidence_codes": ",".join(evidence_codes),
                "source_raw_term_ids": ",".join(raw_term_ids),
                "source_has_allowed_evidence": has_allowed_evidence,
                "source_has_disallowed_evidence": has_disallowed_evidence,
                "probable_reason": probable_reason(
                    row["category"],
                    source_presence,
                    has_allowed_evidence,
                    has_disallowed_evidence,
                ),
            }
        )
        enriched_rows.append(row)
    return pd.DataFrame(enriched_rows)

In [ ]:
left_only_df = enrich_mismatch_frame(left_only_df)

right_only_df = enrich_mismatch_frame(right_only_df)
shared_mismatch_df = enrich_mismatch_frame(shared_mismatch_df)
all_mismatches_df = pd.concat(
    [left_only_df, right_only_df, shared_mismatch_df],
    ignore_index=True,
)

In [ ]:
# all_mismatches_df.head()

## Root-cause summary views
These tables should drive the next decision: fix extraction logic, adjust config, or open the richer-source research gate.


In [ ]:
summary_by_category = (
    all_mismatches_df.groupby(["category", "probable_reason"], dropna=False)
    .size()
    .reset_index(name="pair_count")
    .sort_values(["category", "pair_count"], ascending=[True, False])
)

summary_by_source_presence = (
    all_mismatches_df.groupby(["category", "source_presence"], dropna=False)
    .size()
    .reset_index(name="pair_count")
    .sort_values(["category", "pair_count"], ascending=[True, False])
)

evidence_series = all_mismatches_df.assign(
    source_evidence_code=all_mismatches_df["source_evidence_codes"].str.split(",")
).explode("source_evidence_code")
evidence_series = evidence_series[evidence_series["source_evidence_code"].fillna("") != ""]
summary_by_source_evidence = (
    evidence_series.groupby(["category", "source_evidence_code"], dropna=False)
    .size()
    .reset_index(name="pair_count")
    .sort_values(["category", "pair_count"], ascending=[True, False])
)


print(summary_by_category)
print("============")
print(summary_by_source_presence)
print("============")
print(summary_by_source_evidence)

## Write targeted raw Swiss-Prot text snippets
This is the manual lookup artifact for mismatch proteins only.


In [ ]:
snippets_path = output_dir / "mismatch_swissprot_snippets.txt"
with snippets_path.open("w", encoding="utf-8") as handle:
    for protein_id in sorted(mismatch_protein_ids):
        subset = all_mismatches_df[all_mismatches_df["protein_id"] == protein_id].copy()
        subset = subset.sort_values(["category", "term_id"])
        handle.write(f"===== {protein_id} =====\n")
        category_counts = Counter(subset["category"])
        handle.write(f"Category counts: {dict(category_counts)}\n")
        handle.write("Mismatch rows:\n")
        if subset.empty:
            handle.write("<none>\n")
        else:
            handle.write(subset.to_csv(sep="\t", index=False))
        handle.write("Swiss-Prot raw record:\n")
        handle.write(raw_text_by_protein.get(protein_id, "<NO RAW RECORD FOUND>\n"))
        handle.write("\n\n")

print({"snippets_path": str(snippets_path), "snippet_proteins": len(mismatch_protein_ids)})